In [ ]:
knitr::opts_chunk$set(echo = FALSE,message = FALSE,warning = FALSE)
options(scipen=999)


In [ ]:
if (!exists("book_load_packages", mode = "function")) source("scripts/book_runtime.R")
book_load_packages(fredr,
               dplyr,
               tidyr,
               lubridate,
               caret,
               partykit,
               pROC,
               devtools,
               httr,
               DT,
               httr,
               visNetwork,
               dkstat,
               data.table,
               pROC,
               igraph,
               visNetwork,
               rPref,
               progress,
               quantmod,
               rvest,
               ggrepel)
source(file = "money_theme.R")
source(file = "scripts/analysis/correlations.R")
book_correlation_namespace = "indicators-correlations"


# Leading Economic Indicators


In [ ]:
if (file.exists("Renviron.site")) readRenviron(path = "Renviron.site")
if (.book_runtime$mode != "replay") fredr::fredr_set_key(key = Sys.getenv("FRED_API"))


## Prepare ETF price observations


In [ ]:
df_etfs = openxlsx::read.xlsx(xlsxFile = "Chapters/01_Indicators/etf_list.xlsx")

df_etf_yahoo = rbindlist(lapply(df_etfs$Ticker,function(ticker) {

      temp_df = tryCatch(expr = { book_yahoo(ticker) %>%
      as.data.frame(.) %>%
      dplyr::mutate(Date=row.names(.)) %>%
      dplyr::mutate(Date=gsub("X","",Date)) %>%
      dplyr::mutate(Date=ymd(Date)) %>%
      dplyr::select(c(7,1)) %>%
      setNames(.,c("date","value")) %>%
      dplyr::mutate(series_id=ticker)# %>%
      # tidyr::complete(date=seq.Date(from=min(date,na.rm=T),to=book_date(),by="1 day")) %>%
      # tidyr::fill(value,.direction = "down")  %>%
      # tidyr::fill(series_id,.direction = "down")


      },error=function(e) {
        warning(sprintf("Skipping optional Yahoo series %s: %s", ticker, conditionMessage(e)),
                call. = FALSE)
        data.frame()
      })
      row.names(temp_df) = NULL
      return( temp_df )
}))

if (!nrow(df_etf_yahoo) || all(is.na(df_etf_yahoo$value))) {
  stop("No current ETF price observations are available", call. = FALSE)
}


## Data sources and preparation {#data}


In [ ]:
# BYGV80_meta <- book_dst_metadata(table = "BYGV80", lang = "da")
BYGV80 <- book_dst(table = "BYGV80",
                       BYGFASE="*",
                       ANVENDELSE="*",
                       Tid="*",
                       lang = "da") %>%
  group_by(BYGFASE,ANVENDELSE) %>%
  arrange(BYGFASE,ANVENDELSE,TID) %>%
  filter(ANVENDELSE %in% c("Parcelhuse")) %>%
  ungroup() %>%
  dplyr::select(BYGFASE,
                TID,
                value) %>%
  dplyr::rename(series_id=BYGFASE,
                date=TID)


# BYG1_meta <- book_dst_metadata(table = "BYG1", lang = "da")
BYG1 <- book_dst(table = "BYG1",
                       BRANCHE07="*",
                       SÆSON="*",
                       ART="*",
                       Tid="*",
                       lang = "da") %>%
  filter(SÆSON=="Sæsonkorrigeret") %>%
  dplyr::mutate(filler="BYG1") %>%
  unite(series_id,filler,BRANCHE07,ART,sep = "_") %>%
  dplyr::select(series_id,
                TID,
                value) %>%
  dplyr::rename(date=TID)

# PRIS90_meta <- book_dst_metadata(table = "PRIS90", lang = "da")
PRIS90 <- book_dst(table = "PRIS90",
                       ENHED="*",
                       BOLTYP="*",
                       Tid="*",
                       lang = "da") %>%
  dplyr::rename(date=TID) %>%
  filter(ENHED=="Indeks")  %>%
  unite(BOLTYP,BOLTYP,ENHED) %>%
  dplyr::rename(series_id=BOLTYP)

# BYG42_meta <- book_dst_metadata(table = "BYG42", lang = "da")
BYG42 <- book_dst(table = "BYG42",
                       HINDEKS="*",
                       DINDEKS="*",
                       ART="*",
                       TAL="*",
                       Tid="*",
                       lang = "da") %>%
  dplyr::rename(date=TID) %>%
  filter(DINDEKS=="Byggeomkostningsindeks i alt") %>%
  filter(TAL=="Indeks") %>%
  filter(ART %in% c("Materialer","Arbejdsomkostninger")) %>%
  dplyr::mutate(filler="BYG42") %>%
  unite(series_id,filler,HINDEKS,DINDEKS,ART,sep="_")%>%
  dplyr::select(series_id,
                date,
                value)


## FinansDanmark


In [ ]:
source_url = "https://finansdanmark.dk/tal-og-data/boligstatistik/obligationsrenter/"
xlsx_url = finansdanmark_xlsx_url(source_url)

df_interest = book_workbook(url = xlsx_url,startRow = 1)
df_interest$År[1] = 1997
df_interest = df_interest %>% fill(År,.direction = "down")
df_interest$Date = book_iso_week_monday(df_interest$År, df_interest$Uge)
df_interest = df_interest %>%
  dplyr::select(Date,Kort.rente,Lang.rente) %>%
  dplyr::mutate( CurveLongShort = Lang.rente - Kort.rente ) %>%
  gather(series_id,value,CurveLongShort,Lang.rente,Kort.rente) %>%
  dplyr::rename(date=Date) %>%
  dplyr::select(date,value,series_id)


## FRED


In [ ]:
series_ids = unique(c(# "T10Y2Y",
               "T5YIFR",
               "BAA10Y",
               # "T10Y3M",
               "CPIAUCSL",
               "DFF",
               "ACTLISCOUUS",
               "MORTGAGE30US",
               "COMPUTSA",
               # "RRVRUSQ156N",
               "UMCSENT",
               # "BAMLH0A0HYM2",
               # "RHORUSQ156N",
               "T10YIE",
               "HOUST1F",
               # "BAMLH0A0HYM2",
               "COMPU1USA",
               "M2V",
               # "BAMLC0A4CBBB",
               "PCEPI",
               "AUTHNOTT",
               "PSAVERT",
               # "RRPONTSYD",
               "PERMIT",
               "M1SL",
               "PERMIT1",
               # "T10Y3M",
               "INDPRO",
               # "BAMLC0A4CBBB",
               # "NFCI",
               "CORESTICKM159SFRBATL",
               "PAYEMS",
               "CIVPART",
               "MDSP",
               "M2SL",
               "UNRATE",
               "AUTHNOT1U",
               "HOUST",
               "ICSA",
               "DGS10",
               "DGS20",
               "DGS30",
               "DGS5",
               "DGS2",
               "DGS1",
               "CSUSHPINSA",
               "GDP",
               # "GDPC1",
               # "DFII10",
               # "TDSP",
               "WALCL",
               "MSPUS"))

df_macro_fred = rbindlist(lapply(series_ids,function(series_id) {
  book_fred(series_id = series_id) %>%
  dplyr::select(date,value) %>%
    dplyr::mutate(series_id=series_id) #%>%
    # tidyr::complete(date=seq.Date(from=min(date,na.rm=T),to=book_date(),by="1 day")) %>%
    # tidyr::fill(value,.direction = "down")  %>%
    # tidyr::fill(series_id,.direction = "down")
}))


## Yahoo


In [ ]:
series_ids_yahoo = unique(c("GC=F",
                            "SI=F",
                            "DX-Y.NYB",
                            "BTC-USD",
                            "PA=F",
                            "^IXIC",
                            "^GSPC",
                            "^OMXC25",
                            "USDDKK=X"))

df_macro_yahoo = rbindlist(lapply(series_ids_yahoo,function(ticker) {
      temp_df = book_yahoo(ticker) %>%
      as.data.frame(.) %>%
      dplyr::mutate(Date=row.names(.)) %>%
      dplyr::mutate(Date=gsub("X","",Date)) %>%
      dplyr::mutate(Date=ymd(Date)) %>%
      dplyr::select(c(7,1)) %>%
      setNames(.,c("date","value")) %>%
      dplyr::mutate(series_id=ticker) #%>%
      # tidyr::complete(date=seq.Date(from=min(date,na.rm=T),to=book_date(),by="1 day")) %>%
      # tidyr::fill(value,.direction = "down")  %>%
      # tidyr::fill(series_id,.direction = "down")

      row.names(temp_df) = NULL
      return( temp_df )
})) %>%
  bind_rows( df_etf_yahoo )


## Assemble source series {#combine}

## Validate observations and remove identical repeats


In [ ]:
df_all = df_macro_yahoo %>%
  bind_rows( df_macro_fred ) %>%
  bind_rows(df_interest) %>%
  bind_rows(BYG42) %>%
  bind_rows(PRIS90) %>%
  bind_rows(BYGV80)
df_all = book_prepare_correlations(df_all, namespace = book_correlation_namespace)
all_ids = unique(df_all$series_id)

rm(df_macro_yahoo)
rm(df_macro_fred)
rm(df_etf_yahoo)
gc()


## Align every directed series pair {#combine-1}


In [ ]:
# Alignment matches shared dates and omits pairs of missing values, without
# adding dates, filling observations, changing prices, or resampling series.


## Cross correlations

Cross correlation measures association, not causation. A leading lag does not establish a causal effect or a reliable forecast. The 30, 365 and 1,825 horizons count aligned observations, which are not necessarily calendar days for mixed-frequency sources. Comparisons with insufficient overlap or constant values remain in the results with undefined (`NA`) correlations.


In [ ]:
df_ccfs = book_correlations(df_all, namespace = "indicators-correlations")
book_capture_correlation(book_correlation_namespace, df_all, df_ccfs, df_etfs$Ticker)


In [ ]:
df_best_corr = df_ccfs %>%
  ungroup() %>%
  dplyr::select(x1,x2,best_corr_time_30,best_corr_time_365,best_corr_time_1825) %>%
  gather(corr_time,corr_value, best_corr_time_30, best_corr_time_365, best_corr_time_1825 ) %>%
  arrange(x1,x2) %>%
  group_by(x1,x2) %>%
  filter(abs(corr_value)==max(abs(corr_value))) %>%
  dplyr::mutate(corr_time_min=as.numeric(gsub("best_corr_time_","",corr_time))) %>%
  group_by(x1,x2,corr_value) %>%
  filter(corr_time_min==min(corr_time_min)) %>%
  ungroup() %>%
  dplyr::select(x1,x2,corr_time)

df_best_corr = df_ccfs %>%
  ungroup() %>%
  gather(corr_time,corr_value, best_corr_time_30, best_corr_time_365, best_corr_time_1825 )  %>%
  gather(corr_time_best,corr_time_value, best_corr_value_30, best_corr_value_365, best_corr_value_1825 ) %>%
  inner_join(df_best_corr) %>%
  filter(gsub("best_corr_time_","",corr_time) == gsub("best_corr_value_","",corr_time_best)) %>%
  dplyr::select(x1,x2,corr_value,corr_time_value)


In [ ]:
df_best_corr %>%
  ggplot(.,aes(x=corr_value,y=corr_time_value)) +
  geom_point() +
  geom_vline(xintercept = 0,linetype=2) +
  geom_hline(yintercept = 0,linetype=2) +
  scale_y_continuous(limits = c(-1,1),breaks = seq(-1,1,by=0.1)) +
  scale_x_continuous(breaks = seq(-10,10,by=1)*365,labels=seq(-10,10,by=1))


## Descriptive

## Leading


In [ ]:
df_best_corr %>%
  filter(corr_time_value>0.8) %>%
  filter(corr_value<0-30) %>%
  arrange(corr_value) %>%
  group_by(x1) %>%
  dplyr::summarize(count=n(),
                   median=scales::number(median(corr_value),accuracy=1),
                   corr=scales::number(median(corr_time_value),accuracy=0.01)) %>%
  arrange(desc(count)) %>%
  datatable()


## Coincident


In [ ]:
df_best_corr %>%
  filter(corr_time_value>0.8) %>%
  filter(corr_value>=-30,corr_value<=30) %>%
  arrange(corr_value) %>%
  group_by(x1) %>%
  dplyr::summarize(count=n(),
                   median=scales::number(median(corr_value),accuracy=1),
                   corr=scales::number(median(corr_time_value),accuracy=0.01)) %>%
  arrange(desc(count)) %>%
  datatable()


## Lagging


In [ ]:
df_best_corr %>%
  filter(corr_time_value>0.8) %>%
  filter(corr_value>30) %>%
  arrange(corr_value) %>%
  group_by(x1) %>%
  dplyr::summarize(count=n(),
                   median=scales::number(median(corr_value),accuracy=1),
                   corr=scales::number(median(corr_time_value),accuracy=0.01)) %>%
  arrange(desc(count)) %>%
  datatable()


## Association graph


In [ ]:
normalize_strength <- function(strength, min_width, max_width) {
  normalized_strength <- (strength - min(strength)) / (max(strength) - min(strength))
  arrow_width <- normalized_strength * (max_width - min_width) + min_width
  return(arrow_width)
}


In [ ]:
# Create a data frame representing strong associations
association_structure = df_best_corr %>%
  filter(abs(corr_time_value)>0.9) %>%
  dplyr::rename(from=x1,
                to=x2) %>%
  dplyr::mutate(sign=ifelse(corr_time_value>0,"positive","negative"),
                strength=scales::rescale(abs(corr_time_value),to = c(0,1),from = c(0,1))) %>%
  dplyr::select(from,to,sign,strength)

# Convert the data frame to edge_directions
edge_directions <- as.vector(t(association_structure[, c("from", "to")]))

# Create a graph object with edge_directions
g <- graph(edge_directions, directed = TRUE)

# Create a named vector of colors for positive and negative relationships
edge_colors <- c("positive" = "blue", "negative" = "red")

# Map the colors and thickness using the observed sign and strength
E(g)$color <- edge_colors[association_structure$sign]

normalized_strength <- normalize_strength(association_structure$strength, min_width = 0.5, max_width = 2)
E(g)$width <- normalized_strength

# Plot the association graph with colored and thickness-scaled arrows
plot(g, vertex.label.family = "sans", vertex.label.font = 2, vertex.label.cex = 0.6, edge.arrow.size = 0.2)


## Individual patterns

## DK Long Interest Rate


In [ ]:
df_best_corr %>%
  ungroup() %>%
  filter(x1 %in% c("Lang.rente")) %>%
  filter(abs(corr_time_value) > 0.85)


In [ ]:
df_best_corr %>%
  ungroup() %>%
  filter(x1 %in% c("Lang.rente")) %>%
  filter(abs(corr_time_value) > 0.85) %>%
  dplyr::mutate(corr_time_value=scales::number(corr_time_value,0.01)) %>%
  arrange(desc(corr_time_value)) %>%
  datatable()


In [ ]:
corr_extra = data.frame(x1=all_ids,
                        x2=all_ids,
                        corr_time_value=1)

filtered_cor_df <- df_best_corr %>%
  ungroup() %>%
  filter(x1 %in% c("Lang.rente","CIVPART","DGS20","M2SL","PCEPI","TLT")) %>%
  filter(abs(corr_time_value) > 0.85)

# Create a nodes data frame
nodes <- data.frame(id = unique(c(filtered_cor_df$x1, filtered_cor_df$x2)), label = unique(c(filtered_cor_df$x1, filtered_cor_df$x2)))

# Create an edges data frame
edges <- filtered_cor_df %>%
  mutate(color = if_else(corr_time_value > 0, "green", "red"),
         width = scales::rescale(abs(corr_time_value),to=c(0,30))) %>%
  dplyr::select(from = x1, to = x2, color, width)

# Create a visNetwork graph
visNetwork(nodes, edges) %>%
  visIgraphLayout() %>%
  visOptions(highlightNearest = TRUE, nodesIdSelection = TRUE) %>%
  visInteraction(navigationButtons = TRUE)


## DK Houseprices


In [ ]:
df_best_corr %>%
  ungroup() %>%
  filter(x1 %in% c("Enfamiliehuse_Indeks")) %>%
  filter(abs(corr_time_value) > 0.85) %>%
  dplyr::mutate(corr_time_value=scales::number(corr_time_value,0.01)) %>%
  arrange(desc(corr_time_value)) %>%
  datatable()


In [ ]:
corr_extra = data.frame(x1=all_ids,
                        x2=all_ids,
                        corr_time_value=1)

filtered_cor_df <- df_best_corr %>%
  ungroup() %>%
  filter(x1 %in% c("Enfamiliehuse_Indeks","PCEPI","CSUSHPINSA","M2SL","PCEPI","TLT","AUTHNOTT","Byggeri under opførelse","CurveLongShort","Lang.Rente")) %>%
  filter(abs(corr_time_value) > 0.9)

# Create a nodes data frame
nodes <- data.frame(id = unique(c(filtered_cor_df$x1, filtered_cor_df$x2)), label = unique(c(filtered_cor_df$x1, filtered_cor_df$x2)))

# Create an edges data frame
edges <- filtered_cor_df %>%
  mutate(color = if_else(corr_time_value > 0, "green", "red"),
         width = scales::rescale(abs(corr_time_value),to=c(0,30))) %>%
  dplyr::select(from = x1, to = x2, color, width)

# Create a visNetwork graph
visNetwork(nodes, edges) %>%
  visIgraphLayout() %>%
  visOptions(highlightNearest = TRUE, nodesIdSelection = TRUE) %>%
  visInteraction(navigationButtons = TRUE)

# ChatGPT:

filtered_cor_df %>%
  dplyr::select(-corr_value) %>%
  dplyr::mutate(corr_time_value=scales::number(corr_time_value,accuracy=0.01)) %>%
  dplyr::mutate(label_=paste0(x1,"<->",x2,": ",corr_time_value)) %>%
  pull(label_) %>%
  paste(.,collapse="\n") %>%
  cat(.)


## Employment


In [ ]:
df_best_corr %>%
  ungroup() %>%
  filter(x1 %in% c("PAYEMS")) %>%
  filter(abs(corr_time_value) > 0.85) %>%
  dplyr::mutate(corr_time_value=scales::number(corr_time_value,0.01)) %>%
  arrange(desc(corr_time_value)) %>%
  datatable()


In [ ]:
corr_extra = data.frame(x1=all_ids,
                        x2=all_ids,
                        corr_time_value=1)

filtered_cor_df <- df_best_corr %>%
  ungroup() %>%
  filter(x1 %in% c("PAYEMS","INDPRO","PCEPI","CPIAUCSL","MSPUS","CSUSHPINSA")) %>%
  filter(abs(corr_time_value) > 0.85)

# Create a nodes data frame
nodes <- data.frame(id = unique(c(filtered_cor_df$x1, filtered_cor_df$x2)), label = unique(c(filtered_cor_df$x1, filtered_cor_df$x2)))

# Create an edges data frame
edges <- filtered_cor_df %>%
  mutate(color = if_else(corr_time_value > 0, "green", "red"),
         width = scales::rescale(abs(corr_time_value),to=c(0,30))) %>%
  dplyr::select(from = x1, to = x2, color, width)

# Create a visNetwork graph
visNetwork(nodes, edges) %>%
  visIgraphLayout() %>%
  visOptions(highlightNearest = TRUE, nodesIdSelection = TRUE) %>%
  visInteraction(navigationButtons = TRUE)


## NASDAQ


In [ ]:
df_best_corr %>%
  ungroup() %>%
  filter(x1 %in% c("^IXIC")) %>%
  filter(abs(corr_time_value) > 0.85) %>%
  dplyr::mutate(corr_time_value=scales::number(corr_time_value,0.01)) %>%
  arrange(desc(corr_time_value)) %>%
  datatable()


In [ ]:
corr_extra = data.frame(x1=all_ids,
                        x2=all_ids,
                        corr_time_value=1)

filtered_cor_df <- df_best_corr %>%
  ungroup() %>%
  filter(x1 %in% c("^IXIC","M2SL","GDP","CSUSHPINSA","MSPUS","MSPUS","M1SL","	BTC-USD")) %>%
  filter(abs(corr_time_value) > 0.85)

# Create a nodes data frame
nodes <- data.frame(id = unique(c(filtered_cor_df$x1, filtered_cor_df$x2)), label = unique(c(filtered_cor_df$x1, filtered_cor_df$x2)))

# Create an edges data frame
edges <- filtered_cor_df %>%
  mutate(color = if_else(corr_time_value > 0, "green", "red"),
         width = scales::rescale(abs(corr_time_value),to=c(0,30))) %>%
  dplyr::select(from = x1, to = x2, color, width)

# Create a visNetwork graph
visNetwork(nodes, edges) %>%
  visIgraphLayout() %>%
  visOptions(highlightNearest = TRUE, nodesIdSelection = TRUE) %>%
  visInteraction(navigationButtons = TRUE)


## SPY


In [ ]:
df_best_corr %>%
  ungroup() %>%
  filter(x1 %in% c("^GSPC")) %>%
  filter(abs(corr_time_value) > 0.85) %>%
  dplyr::mutate(corr_time_value=scales::number(corr_time_value,0.01)) %>%
  arrange(desc(corr_time_value)) %>%
  datatable()


In [ ]:
corr_extra = data.frame(x1=all_ids,
                        x2=all_ids,
                        corr_time_value=1)

filtered_cor_df <- df_best_corr %>%
  ungroup() %>%
  filter(x1 %in% c("^GSPC","GDP","M2SL","MSPUS","PCEPI","CSUSHPINSA","CPIAUCSL","WOOD","WALCL")) %>%
  filter(abs(corr_time_value) > 0.85)

# Create a nodes data frame
nodes <- data.frame(id = unique(c(filtered_cor_df$x1, filtered_cor_df$x2)), label = unique(c(filtered_cor_df$x1, filtered_cor_df$x2)))

# Create an edges data frame
edges <- filtered_cor_df %>%
  mutate(color = if_else(corr_time_value > 0, "green", "red"),
         width = scales::rescale(abs(corr_time_value),to=c(0,30))) %>%
  dplyr::select(from = x1, to = x2, color, width)

# Create a visNetwork graph
visNetwork(nodes, edges) %>%
  visIgraphLayout() %>%
  visOptions(highlightNearest = TRUE, nodesIdSelection = TRUE) %>%
  visInteraction(navigationButtons = TRUE)


## Case_Shiller Housing


In [ ]:
df_best_corr %>%
  ungroup() %>%
  filter(x1 %in% c("CSUSHPINSA")) %>%
  filter(abs(corr_time_value) > 0.85) %>%
  dplyr::mutate(corr_time_value=scales::number(corr_time_value,0.01)) %>%
  arrange(desc(corr_time_value)) %>%
  datatable()


In [ ]:
corr_extra = data.frame(x1=all_ids,
                        x2=all_ids,
                        corr_time_value=1)

filtered_cor_df <- df_best_corr %>%
  ungroup() %>%
  filter(x1 %in% c("CSUSHPINSA","GDP","	Enfamiliehuse_Indeks","CPIAUCSL","M2SL","PCEPI","PAYEMS","WOOD","ACTLISCOUUS")) %>%
  filter(abs(corr_time_value) > 0.85)

# Create a nodes data frame
nodes <- data.frame(id = unique(c(filtered_cor_df$x1, filtered_cor_df$x2)), label = unique(c(filtered_cor_df$x1, filtered_cor_df$x2)))

# Create an edges data frame
edges <- filtered_cor_df %>%
  mutate(color = if_else(corr_time_value > 0, "green", "red"),
         width = scales::rescale(abs(corr_time_value),to=c(0,30))) %>%
  dplyr::select(from = x1, to = x2, color, width)

# Create a visNetwork graph
visNetwork(nodes, edges) %>%
  visIgraphLayout() %>%
  visOptions(highlightNearest = TRUE, nodesIdSelection = TRUE) %>%
  visInteraction(navigationButtons = TRUE)
